# Generate Jenkinsfiles (Groovy) from the descriptions with 3 Ollama models

**Before running:** Settings (right panel) → **Accelerator: GPU T4 x2** and **Internet: On**.
Then **Save Version → Save & Run All (Commit)** so it runs on Kaggle's servers even if you close the tab
(about 15–25 minutes, most of it downloading the models).

Output files (Output tab of the version):
- `generated_groovy_for_sheet.csv`: one row per project, one column per model
- `generated_groovy_long.csv`: one row per project × model, with generation time and a basic syntax check
- `outputs/<model>/<no>.groovy`: each generated Jenkinsfile as its own file

## 1. Install Ollama and start the server

In [ ]:
!apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

import subprocess, time, requests, os
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
server = subprocess.Popen(["ollama", "serve"], stdout=open("/kaggle/working/ollama.log", "w"),
                          stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        requests.get("http://127.0.0.1:11434", timeout=2); print("Ollama server is up"); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start - see /kaggle/working/ollama.log")

## 2. Pull the 3 models

In [ ]:
MODELS = ["qwen2.5-coder:7b", "qwen2.5-coder:14b", "deepseek-coder-v2:16b"]
for m in MODELS:
    print("pulling", m, "...")
    r = subprocess.run(["ollama", "pull", m], capture_output=True, text=True)
    print("  ok" if r.returncode == 0 else r.stderr[-500:])
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

## 3. The 5 descriptions (from the sheet)

In [ ]:
ITEMS = [
  {
    "no": 1,
    "title": "Simple Java Maven App",
    "repo": "https://github.com/jenkins-docs/simple-java-maven-app",
    "type": "Declarative pipeline",
    "description": "Write a Jenkins declarative pipeline (Jenkinsfile) for a Java Maven project. It runs on any agent whose label is not 'windows' (label '!windows') and uses the option skipStagesAfterUnstable(). It has three stages: 'Build', which runs the shell command mvn -B -DskipTests clean package; 'Test', which runs mvn test and, in a post 'always' block, publishes the JUnit test reports from target/surefire-reports/*.xml; 'Deliver', which runs the shell script ./jenkins/scripts/deliver.sh."
  },
  {
    "no": 2,
    "title": "Simple Python PyInstaller App",
    "repo": "https://github.com/jenkins-docs/simple-python-pyinstaller-app",
    "type": "Declarative pipeline",
    "description": "Write a Jenkins declarative pipeline (Jenkinsfile) for a small Python command-line app. It runs on any agent and uses the option skipStagesAfterUnstable(). It has three stages: 'Build', which byte-compiles the sources with python3 -m py_compile sources/add2vals.py sources/calc.py; 'Test', which runs py.test --verbose --junit-xml test-reports/results.xml sources/test_calc.py and, in a post 'always' block, publishes the JUnit results from test-reports/results.xml; 'Deliver', which builds a single-file executable with pyinstaller --onefile sources/add2vals.py and, in a post 'success' block, archives the artifact dist/add2vals."
  },
  {
    "no": 3,
    "title": "Multibranch Pipeline Project (Node.js / React)",
    "repo": "https://github.com/jenkins-docs/building-a-multibranch-pipeline-project",
    "type": "Declarative pipeline",
    "description": "Write a Jenkins declarative pipeline (Jenkinsfile) for a Node.js/React app built as a multibranch pipeline. The whole pipeline runs inside a Docker agent using the image 'node:lts-alpine' with the args '-p 3000:3000 -p 5000:5000'. Set the environment variable CI = 'true'. It has four stages: 'Build', which runs npm install; 'Test', which runs ./jenkins/scripts/test.sh; 'Deliver for development', which runs only when the branch is 'development': it runs ./jenkins/scripts/deliver-for-development.sh, then pauses with an input step whose message is 'Finished using the web site? (Click \"Proceed\" to continue)', then runs ./jenkins/scripts/kill.sh; 'Deploy for production', which runs only when the branch is 'production': it runs ./jenkins/scripts/deploy-for-production.sh, then the same input message, then ./jenkins/scripts/kill.sh."
  },
  {
    "no": 4,
    "title": "DevOps Spring Boot (Maven + Docker Hub)",
    "repo": "https://github.com/saravanan81java/devops-springboot",
    "type": "Declarative pipeline",
    "description": "Write a Jenkins declarative pipeline (Jenkinsfile) that builds a Spring Boot app with Maven and pushes a Docker image to Docker Hub. It runs on any agent and uses the Maven tool installation named 'Default'. Set the environment variable PATH to \"/usr/local/bin:$PATH\" so Docker can be found. It has three stages: 'Build Maven', which checks out the code with checkout(scmGit(...)) from the main branch (*/main) of https://github.com/saravanan81java/devops-springboot using the credentials ID 'saravanan81java' and no extensions, then runs mvn clean install; 'Build docker image', which inside a script block runs docker build -t saran81/devops-springboot-integration . ; 'Push image to Hub', which inside a script block uses withCredentials with a secret text credential 'dockerhubpwd' bound to the variable dockerhubpwd to run docker login -u saran81 -p ${dockerhubpwd}, and then runs docker push saran81/devops-springboot-integration."
  },
  {
    "no": 5,
    "title": "Spring Boot Docker Deploy (Scripted Pipeline)",
    "repo": "https://github.com/vdharmaraj/PGDO_Proj3",
    "type": "Scripted pipeline",
    "description": "Write a Jenkins scripted pipeline (Jenkinsfile using node { ... }, not the declarative pipeline block) that builds a Spring Boot app, builds and pushes a Docker image, and redeploys it as a container. Inside node, define the variables application = \"devopsexample\", dockerhubaccountid = \"vikidvg\", mvnHome = tool 'maven-3.5.2', an empty dockerImage, and dockerImageTag = \"${dockerhubaccountid}/${application}:${env.BUILD_NUMBER}\". The stages are: 'Clone Repo', which runs git with url 'https://github.com/vdharmaraj/PGDO_Proj3.git' and branch 'main', then sets mvnHome = tool 'maven-3.5.2' again; 'Build Project', which runs \"'${mvnHome}/bin/mvn' clean install\"; 'Build Docker Image with new code', which sets dockerImage = docker.build(\"${dockerhubaccountid}/${application}:${env.BUILD_NUMBER}\"); 'Push Image to Remote Repo', which echoes \"Docker Image Tag Name ---> ${dockerImageTag}\" and then, inside docker.withRegistry('', 'dockerHub'), pushes the image with the tag \"${env.BUILD_NUMBER}\" and with the tag \"latest\"; 'Remove running container with old code', which runs docker rm -f \\$(docker ps -a -f name=devopsexample -q) || true; 'Deploy Docker Image with new changes', which runs docker run --name devopsexample -d -p 2222:2222 ${dockerhubaccountid}/${application}:${env.BUILD_NUMBER}; 'Remove old images', which runs docker rmi ${dockerhubaccountid}/${application}:latest -f."
  }
]
for it in ITEMS:
    print(f"{it['no']}. {it['title']}  [{it['type']}]  ({len(it['description'].split())} words)")

## 4. Generate the Jenkinsfiles
Same prompt for every model, `temperature = 0` and a fixed seed (repeatable). The model gets only the instruction and the description.

`basic_check` is a quick sanity check, not a real Groovy compile: balanced `{}` `()` `[]` (ignoring strings and comments)
and the file starts with `pipeline {` (declarative) or `node` (scripted).

In [ ]:
import re

SYSTEM = ("You are an experienced DevOps engineer. Write the Jenkinsfile (Jenkins pipeline in Groovy) described by the user. "
          "Follow the description exactly: use the exact stage names, commands, paths, credential IDs and values it gives, "
          "use the pipeline style it asks for (declarative or scripted), do not add anything it does not ask for, "
          "and do not leave anything out. Output only the Jenkinsfile inside a single ```groovy code block, with no explanation.")
OPTIONS = {"temperature": 0, "seed": 42, "num_ctx": 8192, "num_predict": 2048}

def extract_code(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()
    blocks = re.findall(r"```(?:groovy|Groovy|jenkinsfile|Jenkinsfile)?[ \t]*\n(.*?)```", text, flags=re.S)
    if blocks:
        return max(blocks, key=len).strip()
    m = re.search(r"```(?:groovy)?[ \t]*\n(.*)", text, flags=re.S)
    return (m.group(1) if m else text).strip()

def basic_check(src):
    # strip comments and string literals so braces inside them don't count
    s = re.sub(r"/\*.*?\*/", "", src, flags=re.S)
    s = re.sub(r"'''.*?'''|\"\"\".*?\"\"\"", "''", s, flags=re.S)
    s = re.sub(r"'(?:\\.|[^'\\\n])*'|\"(?:\\.|[^\"\\\n])*\"", "''", s)
    s = re.sub(r"//[^\n]*", "", s)
    stack, pairs = [], {")": "(", "]": "[", "}": "{"}
    for ch in s:
        if ch in "([{":
            stack.append(ch)
        elif ch in ")]}":
            if not stack or stack.pop() != pairs[ch]:
                return f"no: unbalanced '{ch}'"
    if stack:
        return f"no: {len(stack)} unclosed bracket(s)"
    body = re.sub(r"^\s*(#!.*\n)?", "", s)
    if not re.match(r"\s*(pipeline\s*\{|node\b)", body):
        return "no: does not start with pipeline { or node"
    return "yes"

results = []
for model in MODELS:
    folder = f"/kaggle/working/outputs/{model.replace(':', '_')}"
    os.makedirs(folder, exist_ok=True)
    for it in ITEMS:
        t = time.time()
        r = requests.post("http://127.0.0.1:11434/api/generate", json={
            "model": model, "system": SYSTEM, "prompt": it["description"],
            "stream": False, "options": OPTIONS}, timeout=3600)
        r.raise_for_status()
        raw = r.json()["response"]
        g = extract_code(raw)
        secs = round(time.time() - t, 1)
        chk = basic_check(g)
        open(f"{folder}/{it['no']}.groovy", "w").write(g)
        open(f"{folder}/{it['no']}.raw.txt", "w").write(raw)
        results.append({"no": it["no"], "title": it["title"], "model": model,
                        "basic_check": chk, "seconds": secs, "generated_groovy": g})
        print(f"{model:24s} #{it['no']}  {secs:6.1f}s  basic check: {chk}")
    requests.post("http://127.0.0.1:11434/api/generate", json={"model": model, "keep_alive": 0})

## 5. Save for the sheet

In [ ]:
import pandas as pd
long = pd.DataFrame(results)
long.to_csv("/kaggle/working/generated_groovy_long.csv", index=False)

wide = long.pivot(index=["no", "title"], columns="model", values="generated_groovy").reset_index()
wide = wide[["no", "title"] + MODELS]
wide.columns = ["No", "Title"] + [f"Generated Groovy ({m})" for m in MODELS]
wide.to_csv("/kaggle/working/generated_groovy_for_sheet.csv", index=False)

display(long[["no", "title", "model", "basic_check", "seconds"]])
print("Saved: generated_groovy_for_sheet.csv and generated_groovy_long.csv (Output tab)")

## 6. Print every generated Jenkinsfile

In [ ]:
for it in ITEMS:
    print("=" * 100)
    print(f"#{it['no']}  {it['title']}")
    for row in results:
        if row["no"] == it["no"]:
            print("-" * 100)
            print(f"[{row['model']}]  basic check: {row['basic_check']}   ({row['seconds']}s)\n")
            print(row["generated_groovy"])